In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2 as cv
import math
from google.colab.patches import cv2_imshow
from PIL import Image as im

In [ ]:
import numpy as np

def convolution2d(image, kernel, stride=1, padding=0):
    """
    Fungsi Konvolusi 2D Manual tanpa Library Pengolahan Citra

    Parameters:
    - image: Matriks citra masukan 2D (grayscale)
    - kernel: Matriks kernel/filter 2D
    - stride: Besarnya pergeseran kernel pada setiap langkah konvolusi
    - padding: Jumlah zero-padding yang ditambahkan di sekeliling citra
    """
    img = image.astype(np.float32)
    k_height, k_width = kernel.shape

    if padding > 0:
        img_padded = np.pad(img, ((padding, padding), (padding, padding)), mode='constant', constant_values=0)
    else:
        img_padded = img.copy()

    p_height, p_width = img_padded.shape

    out_height = int((p_height - k_height) / stride) + 1
    out_width = int((p_width - k_width) / stride) + 1

    output = np.zeros((out_height, out_width), dtype=np.float32)

    for y in range(0, out_height):
        for x in range(0, out_width):
            y_start = y * stride
            y_end = y_start + k_height
            x_start = x * stride
            x_end = x_start + k_width

            roi = img_padded[y_start:y_end, x_start:x_end]

            output[y, x] = np.sum(roi * kernel)

    output = np.clip(output, 0, 255).astype(np.uint8)
    return output

In [ ]:
img = cv.imread('/content/drive/MyDrive/PCVK/Images/mandrill.tiff')
img_gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

def convolution2d(image, kernel, stride=1, padding=1):
    img = image.astype(np.float32)
    k_h, k_w = kernel.shape

    p_h = k_h // 2 if padding == 1 else padding
    p_w = k_w // 2 if padding == 1 else padding

    if p_h > 0 or p_w > 0:
        img_padded = np.pad(img, ((p_h, p_h), (p_w, p_w)), mode='constant', constant_values=0)
    else:
        img_padded = img.copy()

    p_height, p_width = img_padded.shape

    out_h = int((p_height - k_h) / stride) + 1
    out_w = int((p_width - k_w) / stride) + 1
    output = np.zeros((out_h, out_w), dtype=np.float32)

    for y in range(out_h):
        for x in range(out_w):
            y_start, y_end = y * stride, y * stride + k_h
            x_start, x_end = x * stride, x * stride + k_w
            roi = img_padded[y_start:y_end, x_start:x_end]
            output[y, x] = np.sum(roi * kernel)

    return np.clip(output, 0, 255).astype(np.uint8)

def get_gaussian_kernel_21x21(kernel_size=21, sigma=3.0):
    ax = np.linspace(-(kernel_size // 2), kernel_size // 2, kernel_size)
    gauss = np.exp(-0.5 * np.square(ax / sigma))
    kernel_2d = np.outer(gauss, gauss)
    return kernel_2d / np.sum(kernel_2d)

img_gray = cv.imread('/content/drive/MyDrive/PCVK/Images/mandrill.tiff', cv.IMREAD_GRAYSCALE)

if img_gray is None:
    print("Error: File 'mandrill.tiff' tidak ditemukan! Pastikan file berada di direktori kerja.")
else:
    # 4. Definisi Kernel & Operasi Filter
    # - Sharpen
    kernel_sharpen = np.array([
        [ 0, -1,  0],
        [-1,  5, -1],
        [ 0, -1,  0]
    ], dtype=np.float32)

    # - Emboss
    kernel_emboss = np.array([
        [-2, -1,  0],
        [-1,  1,  1],
        [ 0,  1,  2]
    ], dtype=np.float32)

    # - Left Sobel Edge Detection (Pendeteksian tepi arah vertikal/kiri)
    kernel_left_sobel = np.array([
        [ 1,  0, -1],
        [ 2,  0, -2],
        [ 1,  0, -1]
    ], dtype=np.float32)

    # - 21x21 Gaussian Blur
    kernel_gaussian_21 = get_gaussian_kernel_21x21(21, sigma=3.0)

    res_sharpen = convolution2d(img_gray, kernel_sharpen, stride=1, padding=1)
    res_emboss = convolution2d(img_gray, kernel_emboss, stride=1, padding=1)
    res_left_sobel = convolution2d(img_gray, kernel_left_sobel, stride=1, padding=1)
    res_gaussian_21 = convolution2d(img_gray, kernel_gaussian_21, stride=1, padding=1)

    res_canny = cv.Canny(img_gray, threshold1=100, threshold2=200)

    table_data = [
        ("Sharpen", "[[ 0, -1,  0],\n [-1,  5, -1],\n [ 0, -1,  0]]", res_sharpen),
        ("Emboss", "[[-2, -1,  0],\n [-1,  1,  1],\n [ 0,  1,  2]]", res_emboss),
        ("Left Sobel\nEdge Detection", "[[ 1,  0, -1],\n [ 2,  0, -2],\n [ 1,  0, -1]]", res_left_sobel),
        ("Canny\nEdge Detection", "[ Multistage Algorithm ]\n1. Gaussian Blur\n2. Sobel Gradient\n3. Non-Max Suppression\n4. Hysteresis Threshold", res_canny),
        ("21x21 Gaussian Blur", "[ Matriks 21x21 ]\nSigma = 3.0\nKernel Size = 21x21", res_gaussian_21)
    ]

    fig, axes = plt.subplots(len(table_data), 3, figsize=(12, 3.8 * len(table_data)))

    headers = ["Operasi", "Kernel", "Hasil"]

    for col, head in enumerate(headers):
        fig.text(0.22 + col * 0.32, 0.98, head, fontsize=14, fontweight='bold', ha='center')

    for idx, (name, kernel_str, res_img) in enumerate(table_data):
        axes[idx, 0].text(0.5, 0.5, name, fontsize=12, fontweight='bold', ha='center', va='center')
        axes[idx, 0].axis('off')

        axes[idx, 1].text(0.5, 0.5, kernel_str, fontsize=10, family='monospace', ha='center', va='center')
        axes[idx, 1].axis('off')

        axes[idx, 2].imshow(res_img, cmap='gray')
        axes[idx, 2].set_title(name.replace('\n', ' '))
        axes[idx, 2].axis('on')

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

In [ ]:
# Fungsi tampil berdampingan
def show_side_by_side(images, titles, figsize=(15,5)):
    plt.figure(figsize=figsize)
    for i, (img, title) in enumerate(zip(images, titles)):
        if len(img.shape) == 2:  # grayscale
            plt.subplot(1, len(images), i+1)
            plt.imshow(img, cmap="gray")
        else:  # color
            plt.subplot(1, len(images), i+1)
            plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
        plt.title(title)
        plt.axis("off")
    plt.show()

img = cv.imread("/content/drive/MyDrive/PCVK/Images/lena.jpg")
img_gray = cv.cvtColor(img,cv.COLOR_BGR2GRAY)

blur = cv.GaussianBlur(img, (7,7), 1)
edges = cv.Canny(cv.cvtColor(img, cv.COLOR_BGR2GRAY), 100, 200)
sharpen_kernel = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]])
sharpened = cv.filter2D(img, -1, sharpen_kernel)
show_side_by_side([img, blur, sharpened, edges],
                   ["Asli", "Gaussian Blur", "Sharpened", "Canny Edge Detection"])

In [ ]:
# Bilateral Filter (edge-preserving)
bilateral = cv.bilateralFilter(img, 50, 100, 100)

# Edge Preserving Filter (alternatif Guided Filter)
edge_preserve = cv.edgePreservingFilter(img, flags=1, sigma_s=100, sigma_r=0.9)

show_side_by_side([img, bilateral, edge_preserve],
                   ["Asli", "Bilateral Filter", "Edge Preserving Filter"])

In [ ]:
smooth = cv.bilateralFilter(img, d=15, sigmaColor=75, sigmaSpace=75)

gaussian = cv.GaussianBlur(smooth, (0,0), 3)
sharpened = cv.addWeighted(smooth, 1.5, gaussian, -0.5, 0)

alpha = 1.2  # contrast
beta = 15    # brightness
beauty = cv.convertScaleAbs(sharpened, alpha=alpha, beta=beta)

sepia_kernel = np.array([[0.272, 0.534, 0.131],
                         [0.349, 0.686, 0.168],
                         [0.393, 0.769, 0.189]])
sepia = cv.transform(img, sepia_kernel)
sepia = np.clip(sepia, 0, 255).astype(np.uint8)

rows, cols = img.shape[:2]
kernel_x = cv.getGaussianKernel(cols, cols*0.6)
kernel_y = cv.getGaussianKernel(rows, rows*0.6)
kernel = kernel_y * kernel_x.T
mask = kernel / kernel.max()
vignette = np.copy(sepia)
for i in range(3):
    vignette[:,:,i] = vignette[:,:,i] * mask

noise = np.random.normal(0, 15, vignette.shape).astype(np.int16)
old_img = np.clip(vignette.astype(np.int16) + noise, 0, 255).astype(np.uint8)

show_side_by_side([img, beauty, old_img],
                   ["Asli", "Beauty Filter", "Old/Vintage Filter"])

In [ ]:
gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
gray_blur = cv.medianBlur(gray, 7)
edges = cv.adaptiveThreshold(gray_blur, 255,
                             cv.ADAPTIVE_THRESH_MEAN_C,
                             cv.THRESH_BINARY, 9, 9)

color = cv.bilateralFilter(img, d=9, sigmaColor=200, sigmaSpace=200)

cartoon = cv.bitwise_and(color, color, mask=edges)

show_side_by_side([img, cartoon], ["Asli", "Cartoon Filter"])

In [ ]:
img = cv.imread("/content/drive/MyDrive/PCVK/Images/djawatan.jpg")
img_gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)

night = cv.convertScaleAbs(img, alpha=0.6, beta=-40)

blue_tint = np.full_like(night, (50, 0, 100))  # BGR
night = cv.addWeighted(night, 0.8, blue_tint, 0.2, 0)

kernel = np.ones((15,15), np.float32) / 225
glow = cv.filter2D(night, -1, kernel)

night_glow = cv.addWeighted(night, 0.7, glow, 0.3, 0)

show_side_by_side([img, night, night_glow],
                   ["Asli", "Night Filter", "Night + Glow (filter2D)"])

In [ ]:
alpha = 0.9  # contrast
beta = 20    # brightness
soft = cv.convertScaleAbs(img, alpha=alpha, beta=beta)

warm_tint = np.full_like(soft, (40, 70, 120))  # BGR
pagi = cv.addWeighted(soft, 0.8, warm_tint, 0.2, 0)

kernel = cv.getGaussianKernel(3, 3)
kernel = kernel @ kernel.T  # jadikan 2D kernel
kabut = cv.filter2D(pagi, -1, kernel)

white_layer = np.full_like(pagi, 255)
kabut = cv.addWeighted(kabut, 0.7, white_layer, 0.3, 0)

show_side_by_side([img, pagi, kabut],
                   ["Asli", "Pagi", "Pagi + Kabut"])

In [ ]:
import cv2 as cv2

# 1. Baca citra dan ubah ke grayscale
img = cv2.imread('/content/drive/MyDrive/PCVK/Cat.png', cv2.IMREAD_GRAYSCALE)

# 2. (a) Tambahkan Noise Salt and Pepper secara acak
def add_salt_and_pepper_noise(image, amount=0.04):
    noisy_image = image.copy()

    # Salt noise (putih)
    num_salt = np.ceil(amount * image.size * 0.5)
    coords = [np.random.randint(0, i - 1, int(num_salt)) for i in image.shape]
    noisy_image[tuple(coords)] = 255

    # Pepper noise (hitam)
    num_pepper = np.ceil(amount * image.size * 0.5)
    coords = [np.random.randint(0, i - 1, int(num_pepper)) for i in image.shape]
    noisy_image[tuple(coords)] = 0

    return noisy_image

noisy_img = add_salt_and_pepper_noise(img, amount=0.05)

# 3. (b) Penerapan Mean Filter (Blur 3x3) & Median Filter (3x3)
mean_filtered = cv2.blur(noisy_img, (3, 3))
median_filtered = cv2.medianBlur(noisy_img, 3)

# 4. Visualisasi Hasil
titles = ['Citra Asli', 'Citra + Noise Salt & Pepper', 'Mean Filter 3x3', 'Median Filter 3x3']
images = [img, noisy_img, mean_filtered, median_filtered]

plt.figure(figsize=(12, 8))
for i in range(4):
    plt.subplot(2, 2, i + 1)
    plt.imshow(images[i], cmap='gray')
    plt.title(titles[i])
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Kernel Sharpening 3x3
kernel_3x3 = np.array([[ 0, -1,  0],
                       [-1,  5, -1],
                       [ 0, -1,  0]])

# Kernel Sharpening 5x5
kernel_5x5 = np.array([[-1, -1, -1, -1, -1],
                       [-1,  2,  2,  2, -1],
                       [-1,  2,  8,  2, -1],
                       [-1,  2,  2,  2, -1],
                       [-1, -1, -1, -1, -1]]) / 8.0

# Terapkan pada citra buram
sharpen_3x3 = cv.filter2D(img, -1, kernel_3x3)
sharpen_5x5 = cv.filter2D(img, -1, kernel_5x5)

# Tampilkan Hasil Evaluasi
plt.figure(figsize=(12, 6))
plt.subplot(1, 3, 1), plt.imshow(img, cmap='gray'), plt.title('Original Blurry')
plt.subplot(1, 3, 2), plt.imshow(sharpen_3x3, cmap='gray'), plt.title('Sharpen 3x3')
plt.subplot(1, 3, 3), plt.imshow(sharpen_5x5, cmap='gray'), plt.title('Sharpen 5x5')
plt.show()

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

def kartun(image):

    # 1. Konversi ke grayscale dan beri blurring halus untuk mengurangi noise garis
    gray = cv.cvtColor(image, cv.COLOR_BGR2GRAY)
    gray_blur = cv.medianBlur(gray, 7)

    # 2. Deteksi garis tepi menggunakan Adaptive Thresholding
    edges = cv.adaptiveThreshold(
        gray_blur,
        255,
        cv.ADAPTIVE_THRESH_MEAN_C,
        cv.THRESH_BINARY,
        blockSize=9,
        C=9
    )

    # 3. Penghalusan warna menggunakan Bilateral Filter (menjaga batas warna tetap tajam)
    color = cv.bilateralFilter(image, d=9, sigmaColor=200, sigmaSpace=200)

    # 4. Penggabungan warna dan garis tepi menggunakan operasi logika (bitwise_and)
    cartoon_img = cv.bitwise_and(color, color, mask=edges)

    return cartoon_img

# --- TAMPILKAN BERDAMBINGAN ---

# 1. Baca citra masukan
img = cv.imread("/content/drive/MyDrive/PCVK/Cat.png")  # Sesuaikan path citra Anda

if img is None:
    print("Gagal memuat gambar! Periksa kembali path file gambar Anda.")
else:
    # 2. Panggil fungsi kustom kartun
    img_kartun = kartun(img)

    # 3. Visualisasi Hasil Berdampingan (Asli vs Kartun)
    plt.figure(figsize=(12, 6))

    # Citra Asli
    plt.subplot(1, 2, 1)
    plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
    plt.title("Citra Asli")
    plt.axis("off")

    # Citra Kartun
    plt.subplot(1, 2, 2)
    plt.imshow(cv.cvtColor(img_kartun, cv.COLOR_BGR2RGB))
    plt.title("Hasil Filter Kartun")
    plt.axis("off")

    plt.tight_layout()
    plt.show()